# 보이스피싱 분석 데이터셋 구축

Google Drive의 전사 결과, 정상 금융상담 데이터, 우체국 피해사례를 읽어 분석·머신러닝·대시보드용 테이블을 생성합니다. GPU는 필요하지 않습니다.

In [ ]:
!pip -q install pandas pyarrow openpyxl

## 1. Google Drive 연결 및 경로 설정

In [ ]:
from google.colab import drive, files
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
RESULT_ROOT = PROJECT_ROOT / '분석 결과'
normal_candidates = [
    RESULT_ROOT / '00.금융분야 고객상담 데이터',
    PROJECT_ROOT / '00.금융분야 고객상담 데이터',
]
NORMAL_ROOT = next((path for path in normal_candidates if path.exists()), normal_candidates[0])
EXTERNAL_ROOT = PROJECT_ROOT / '외부 데이터'
OUTPUT_ROOT = PROJECT_ROOT / '구축 데이터셋'

print('보이스피싱 JSON:', len(list(RESULT_ROOT.rglob('cases.json'))))
print('정상상담 ZIP:', len(list(NORMAL_ROOT.rglob('*.zip'))) if NORMAL_ROOT.exists() else 0)
print('저장 위치:', OUTPUT_ROOT)
assert RESULT_ROOT.exists(), f'분석 결과 경로를 확인하세요: {RESULT_ROOT}'
assert list(RESULT_ROOT.rglob('cases.json')), 'cases.json을 찾지 못했습니다.'

## 2. 구축 스크립트와 우체국 CSV 확인

Drive 안에서 파일을 자동으로 찾습니다. 없을 때만 파일 선택 창이 열립니다.

In [ ]:
script_candidates = list(PROJECT_ROOT.rglob('build_voicephishing_datasets*.py'))
if script_candidates:
    SCRIPT_PATH = max(script_candidates, key=lambda path: path.stat().st_mtime)
else:
    print('build_voicephishing_datasets.py를 선택하세요.')
    uploaded = files.upload()
    assert 'build_voicephishing_datasets.py' in uploaded, '올바른 스크립트를 선택해야 합니다.'
    SCRIPT_PATH = PROJECT_ROOT / 'build_voicephishing_datasets.py'
    SCRIPT_PATH.write_bytes(uploaded['build_voicephishing_datasets.py'])

postal_candidates = list(PROJECT_ROOT.rglob('df_postal*.csv'))
if postal_candidates:
    POSTAL_PATH = max(postal_candidates, key=lambda path: path.stat().st_mtime)
else:
    print('df_postal.csv를 선택하세요. 없으면 이 셀을 중지하고 Drive에 넣은 뒤 다시 실행하세요.')
    uploaded = files.upload()
    assert 'df_postal.csv' in uploaded, 'df_postal.csv를 선택해야 합니다.'
    EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)
    POSTAL_PATH = EXTERNAL_ROOT / 'df_postal.csv'
    POSTAL_PATH.write_bytes(uploaded['df_postal.csv'])

print('구축 스크립트:', SCRIPT_PATH)
print('우체국 피해사례:', POSTAL_PATH)

## 3. 데이터셋 생성

In [ ]:
import subprocess
import sys

command = [
    sys.executable, str(SCRIPT_PATH),
    '--result-root', str(RESULT_ROOT),
    '--postal-path', str(POSTAL_PATH),
    '--output-root', str(OUTPUT_ROOT),
]
if NORMAL_ROOT.exists():
    command.extend(['--normal-root', str(NORMAL_ROOT)])

subprocess.run(command, check=True)

## 4. 생성 결과 자동 검증

In [ ]:
import json
import pandas as pd
from IPython.display import display

report_path = OUTPUT_ROOT / '04_reports' / 'validation_report.json'
manifest_path = OUTPUT_ROOT / '04_reports' / 'dataset_manifest.json'
assert report_path.exists(), f'검증 보고서가 없습니다: {report_path}'
report = json.loads(report_path.read_text(encoding='utf-8'))
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
manifest_df = pd.DataFrame(manifest)[['name', 'rows', 'columns', 'csv', 'parquet']]
display(manifest_df)

for table_name, result in report.items():
    assert result.get('duplicate_id_count', 0) == 0, f'{table_name}: ID 중복 발생'
    assert result.get('missing_id_count', 0) == 0, f'{table_name}: ID 누락 발생'
assert report['vp_files']['rows'] == len(list(RESULT_ROOT.rglob('cases.json'))), '입력 JSON과 파일 테이블 수가 다릅니다.'
assert report['vp_cases']['rows'] > 0, '사건 테이블이 비어 있습니다.'
assert report['vp_utterances']['rows'] > 0, '발화 테이블이 비어 있습니다.'
assert 'vp_amount_events' in report, '금액 이벤트 테이블이 생성되지 않았습니다.'
assert report['우체국_피해사례_표준화']['rows'] > 0, '우체국 표준화 테이블이 비어 있습니다.'
assert report['사기유형_통합비교']['rows'] > 0, '통합 비교 테이블이 비어 있습니다.'
print('기본 ID·입력 수·추가 비교표 검증 완료')
if report['normal_finance_calls']['rows'] == 0:
    print('주의: 정상상담 데이터가 없어 정상·사기 이진분류 학습은 아직 할 수 없습니다.')

## 5. 주요 테이블 미리보기

In [ ]:
STANDARD_ROOT = OUTPUT_ROOT / '01_standard_tables'
ML_ROOT = OUTPUT_ROOT / '02_ml_tables'
DASHBOARD_ROOT = OUTPUT_ROOT / '03_dashboard_tables'
preview_names = [
    ('사건 기본표', STANDARD_ROOT / 'vp_cases.csv'),
    ('사칭 대상', STANDARD_ROOT / 'vp_impersonations.csv'),
    ('금액 이벤트', STANDARD_ROOT / 'vp_amount_events.csv'),
    ('ML 탐지', ML_ROOT / 'fraud_detection_ml.csv'),
    ('보이스피싱 한글 사건요약', DASHBOARD_ROOT / '보이스피싱_사건요약_한글.csv'),
    ('우체국 피해사례 표준화', DASHBOARD_ROOT / '우체국_피해사례_표준화.csv'),
    ('사기유형 통합비교', DASHBOARD_ROOT / '사기유형_통합비교.csv'),
]
for name, path in preview_names:
    if path.exists():
        table_df = pd.read_csv(path, encoding='utf-8-sig')
        print(f'\n{name}: {len(table_df):,}행 × {len(table_df.columns):,}열')
        display(table_df.head(3))

## 핵심 해석 주의사항

- `우체국_피해사례_표준화`의 피해액은 실제 피해자료입니다.
- `vp_amount_events`는 통화 금액을 단순 언급·요구·합의·이체완료 주장으로 구분합니다.
- `보이스피싱_사건요약_한글`의 언급·요구·합의·이체주장 금액은 실제 피해액이 아닙니다.
- 두 금액은 `사기유형_통합비교`에서도 서로 다른 열로 유지되므로 합쳐서 계산하면 안 됩니다.
- 자동 추출한 사칭·요구행동·심리전략은 `SILVER` 라벨이므로 과제 보고서에 한계를 명시합니다.